In [ ]:
성능을 더 높이는 필승 전략: "단가(Unit Price)" 피처 추가 이미 **Value(금액)**와 **Weight(중량)**가 모두 중요하다고 검증하셨습니다. 그렇다면 이 둘의 관계인 **"단가(Price = Value / Weight)"**가 바로 Hidden Key입니다.

경제학적으로 **가격(단가)**은 수요를 결정하는 가장 강력한 요인입니다.

시나리오 A: 지난달에 수출 금액(Value)은 그대로인데 중량(Weight)이 줄었다? → 단가가 비싸졌다(Price Up) → 다음 달 수요 감소 가능성.

시나리오 B: 수출 금액(Value)이 늘었는데 중량(Weight)이 폭증했다? → 단가가 싸졌다(Price Down) → 박리다매 추세.

모델이 Value와 Weight를 각각 보는 것보다, 이 둘을 나눈 Price 정보를 명시적으로 주면 성능이 무조건 오릅니다. 기존 코드의 build_training_data_optimized 함수 안에 단가 관련 로직만 추가하면 됩니다.

수정 포인트 단가 계산: Price = Value / (Weight + 1) (0으로 나누기 방지)

단가 변동성: 단가가 급격히 변했는지 확인 (Price_Change)

고가품 여부: 이 품목이 '비싼 물건'인지 '싼 물건'인지 판단 이 밑에 있는 코드가 가장 높게 나온 코드입니다.

# 0.3714743601

In [ ]:
"""
무역 데이터 공행성 예측 모델 (Weight 피처 포함)
- 기존 28개 피처 + 조건부 Weight 4개 = 총 32개 피처
- 검증 결과: Value 클 때 Weight 유용 (상관 0.640)
- ⭐ 최종 예측: LGBM 70% + XGBoost 30% 앙상블 ⭐
"""

import datetime
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor       # ⭐ 추가됨
from tqdm import tqdm

# ============================================================================
# 데이터 로드 및 전처리
# ============================================================================

print("="*80)
print("데이터 로드 시작")
print("="*80)

# 월별 집계 (Value + Weight)
monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)
    .agg({
        'value': 'sum',
        'weight': 'sum'
    })
)

monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

# Value pivot
pivot = (
    monthly
    .pivot(index="item_id", columns="ym", values="value")
    .fillna(0.0)
)

# Weight pivot (조건부 피처용)
pivot_weight = (
    monthly
    .pivot(index="item_id", columns="ym", values="weight")
    .fillna(0.0)
)

months_dt = pivot.columns.to_list()

print(f"✓ 데이터 로드 완료")
print(f"  - 품목 수: {len(pivot)}")
print(f"  - 기간: {len(months_dt)}개월")
print(f"  - Value pivot: {pivot.shape}")
print(f"  - Weight pivot: {pivot_weight.shape}")

# ============================================================================
# 조건부 Weight 피처 함수
# ============================================================================

def create_conditional_weight_features(value, weight, value_series):
    """
    검증 결과 기반 조건부 Weight 피처 생성

    검증 결과:
    - Value 작을 때: Weight-Value 상관 0.254 (약함)
    - Value 클 때: Weight-Value 상관 0.640 (강함)
    → Value 클 때만 Weight 강조!
    """
    value_median = np.median(value_series[value_series > 0]) if (value_series > 0).any() else 0
    value_q75 = np.percentile(value_series[value_series > 0], 75) if (value_series > 0).any() else 0

    # 1. Value 클 때만 Weight 강조
    if value > value_median:
        weight_for_high_value = weight * 1.5
    else:
        weight_for_high_value = 0

    # 2. Value 비례 가중치
    weight_value_ratio = weight * (value / (value_median + 1))

    # 3. 지수 가중 (Value 클수록 증가)
    exp_weighted = weight * (1 - np.exp(-value / (value_median + 1)))

    # 4. 구간별 차등 가중치
    if value > value_q75:
        weight_importance = weight * 1.5  # 상위 25%
    elif value > value_median:
        weight_importance = weight * 1.0  # 중간
    else:
        weight_importance = weight * 0.1  # 하위

    return {
        'weight_for_high_value': weight_for_high_value,
        'weight_value_ratio': weight_value_ratio,
        'exp_weighted': exp_weighted,
        'weight_importance': weight_importance
    }
def create_enhanced_price_features_v2(value, weight, value_series, weight_series):
    """
    Weight 피처의 성공 패턴을 Price에도 적용
    """
    eps = 1.0
    price_t = value / (weight + eps)

    past_prices = []
    for i in range(len(value_series)):
        if weight_series[i] > 0:
            past_prices.append(value_series[i] / (weight_series[i] + eps))

    if len(past_prices) == 0:
        return {
            'price_for_high_value': 0,
            'price_value_interaction': 0,
            'price_volatility_conditional': 0,
            'price_trend_strength': 0,
            'price_volume_pattern': 0
        }

    past_prices = np.array(past_prices)

    # Value 기준 통계
    value_median = np.median(value_series[value_series > 0]) if (value_series > 0).any() else 0
    value_q75 = np.percentile(value_series[value_series > 0], 75) if (value_series > 0).any() else 0

    # Price 기준 통계
    price_mean = np.mean(past_prices)
    price_std = np.std(past_prices)
    price_median = np.median(past_prices)

    # 1. Value 클 때만 Price 강조
    if value > value_median:
        price_for_high_value = price_t * 2.0
    else:
        price_for_high_value = 0

    # 2. Price-Value 상호작용 (조건부)
    if value > value_q75:
        price_value_interaction = (price_t / (price_mean + 0.01)) * (value / (value_median + 1)) * 2.0
    elif value > value_median:
        price_value_interaction = (price_t / (price_mean + 0.01)) * (value / (value_median + 1))
    else:
        price_value_interaction = 0

    # 3. 조건부 가격 변동성
    price_cv = price_std / (price_mean + 0.01)
    if value > value_median:
        price_volatility_conditional = price_cv * (value / value_median)
    else:
        price_volatility_conditional = 0

    # 4. 가격 트렌드 강도
    if len(past_prices) >= 6:
        recent_6 = past_prices[-6:]
        x = np.arange(len(recent_6))
        slope = np.polyfit(x, recent_6, 1)[0]
        price_trend_strength = slope / (price_mean + 0.01)
    else:
        price_trend_strength = 0

    # 5. 가격-거래량 패턴
    weight_median = np.median(weight_series[weight_series > 0]) if (weight_series > 0).any() else 1

    if price_t > price_median and weight < weight_median and value > value_median:
        price_volume_pattern = (price_t / price_median) * (value / value_median) / (weight / weight_median + 0.1)
    elif price_t < price_median and weight > weight_median and value < value_median:
        price_volume_pattern = -0.5
    else:
        price_volume_pattern = 0

    return {
        'price_for_high_value': price_for_high_value,
        'price_value_interaction': price_value_interaction,
        'price_volatility_conditional': price_volatility_conditional,
        'price_trend_strength': price_trend_strength,
        'price_volume_pattern': price_volume_pattern
    }

def create_weight_price_interaction_features(value, weight, value_series, weight_series):
    """
    Weight와 Price의 결합 피처
    """
    eps = 1.0
    price_t = value / (weight + eps)

    value_median = np.median(value_series[value_series > 0]) if (value_series > 0).any() else 0
    weight_median = np.median(weight_series[weight_series > 0]) if (weight_series > 0).any() else 1

    past_prices = []
    for i in range(len(value_series)):
        if weight_series[i] > 0:
            past_prices.append(value_series[i] / (weight_series[i] + eps))
    price_mean = np.mean(past_prices) if len(past_prices) > 0 else 1

    # 1. Value 대비 Weight-Price 효율성
    if value > value_median:
        efficiency = (value / (value_median + 1)) * (weight / (weight_median + 1)) * (price_t / (price_mean + 0.01))
    else:
        efficiency = 0

    # 2. 단가 급변 신호
    if len(past_prices) >= 3:
        recent_3 = past_prices[-3:]
        if len(recent_3) == 3:
            price_acceleration = (recent_3[-1] - recent_3[-2]) - (recent_3[-2] - recent_3[-3])
            price_shock = abs(price_acceleration) / (price_mean + 0.01)
        else:
            price_shock = 0
    else:
        price_shock = 0

    # 3. Value-Weight-Price 삼중 상호작용
    if value > value_median and weight > weight_median and price_t > price_mean:
        triple_interaction = (value / value_median) * (weight / weight_median) * (price_t / price_mean)
    else:
        triple_interaction = 0

    return {
        'wp_efficiency': efficiency,
        'wp_price_shock': price_shock,
        'wp_triple_interaction': triple_interaction
    }
# ============================================================================
# 학습 데이터 구축 (32개 피처)
# ============================================================================

def build_training_data_optimized(pivot, pivot_weight, pairs, months_dt):
    """
    기존 28개 피처 + 조건부 Weight 4개 = 총 32개 피처
    """
    months = months_dt
    n_months = len(months)
    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        # Pivot에 존재하는지 확인
        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        # Value 시계열
        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # Weight 시계열
        a_weight_series = pivot_weight.loc[leader].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        for t in range(max(lag, 12), n_months - 1):
            # Value 변수
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            b_t_2 = b_series[t - 2]
            b_t_12 = b_series[t - 12]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]

            # Weight 변수
            b_weight_t = b_weight_series[t]

            target_month = months[t + 1].month

            # ========== 기존 28개 피처 ==========

            # 1. 선행 품목(A) 트렌드 (3개)
            a_t_lag_diff = a_series[t - lag] - a_series[t - lag - 1]
            a_momentum = a_series[t - lag] / (a_series[t - lag - 3] + 1)
            a_roll_mean_3 = (a_series[t-lag] + a_series[t-lag-1] + a_series[t-lag-2]) / 3

            # 2. 후행 품목(B) 트렌드 (6개)
            b_diff = b_t - b_t_1
            b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
            b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
            b_roll_mean_3 = (b_series[t] + b_series[t-1] + b_series[t-2]) / 3
            b_roll_mean_6 = (b_series[t-5:t+1]).mean()
            b_roll_std_3 = np.std([b_series[t], b_series[t-1], b_series[t-2]])

            # 3. A-B 상호작용 (3개)
            ab_ratio = a_t_lag / (b_t + 1)
            ab_diff = a_t_lag - b_t
            corr_weighted_a = a_t_lag * corr

            # 4. 추가 고급 피처 (5개)
            b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
            b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
            b_yoy_ratio = b_t / (b_t_12 + 1)
            b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
            recent_12 = b_series[max(0, t-11):t+1]
            b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

            # 5. 계절성 (2개)
            quarter = (target_month - 1) // 3 + 1
            is_year_end = 1 if target_month in [11, 12, 1] else 0

            # 6. Lag 가중치 (1개)
            lag_weight = 1 / (1 + lag)

            # ========== 조건부 Weight 피처 (4개) ==========
            b_weight_features = create_conditional_weight_features(
                value=b_t,
                weight=b_weight_t,
                value_series=b_series
            )
            #개선된 price 5개
            b_price_features = create_enhanced_price_features_v2(
                value=b_t,
                weight=b_weight_t,
                value_series=b_series[:t+1],  # 현재까지의 이력
                weight_series=b_weight_series[:t+1]
            )
            b_wp_features = create_weight_price_interaction_features(
                value=b_t,
                weight=b_weight_t,
                value_series=b_series[:t+1],
                weight_series=b_weight_series[:t+1]
            )


            # 데이터 행 생성
            rows.append({
                # 기본 (7개)
                "b_t": b_t,
                "b_t_1": b_t_1,
                "b_t_12": b_t_12,
                "a_t_lag": a_t_lag,
                "max_corr": corr,
                "best_lag": float(lag),
                "month": float(target_month),

                # A 트렌드 (3개)
                "a_t_lag_diff": a_t_lag_diff,
                "a_momentum": a_momentum,
                "a_roll_mean_3": a_roll_mean_3,

                # B 트렌드 (6개)
                "b_diff": b_diff,
                "b_pct_change": b_pct_change,
                "b_yoy_growth": b_yoy_growth,
                "b_roll_mean_3": b_roll_mean_3,
                "b_roll_mean_6": b_roll_mean_6,
                "b_roll_std_3": b_roll_std_3,

                # 상호작용 (3개)
                "ab_ratio": ab_ratio,
                "ab_diff": ab_diff,
                "corr_weighted_a": corr_weighted_a,

                # 추가 고급 피처 (5개)
                "b_cv": b_cv,
                "b_acceleration": b_acceleration,
                "b_yoy_ratio": b_yoy_ratio,
                "b_ma_ratio": b_ma_ratio,
                "b_percentile_rank": b_percentile_rank,

                # 계절성 (2개)
                "quarter": float(quarter),
                "is_year_end": is_year_end,

                # Lag (1개)
                "lag_weight": lag_weight,

                # 조건부 Weight 4개
                "b_weight_for_high_value": b_weight_features['weight_for_high_value'],
                "b_weight_value_ratio": b_weight_features['weight_value_ratio'],
                "b_exp_weighted": b_weight_features['exp_weighted'],
                "b_weight_importance": b_weight_features['weight_importance'],
                # 🔥 Price V2 5개
                "b_price_for_high_value": b_price_features['price_for_high_value'],
                "b_price_value_interaction": b_price_features['price_value_interaction'],
                "b_price_volatility_conditional": b_price_features['price_volatility_conditional'],
                "b_price_trend_strength": b_price_features['price_trend_strength'],
                "b_price_volume_pattern": b_price_features['price_volume_pattern'],
                # 🔥 WP 상호작용 3개
                "b_wp_efficiency": b_wp_features['wp_efficiency'],
                "b_wp_price_shock": b_wp_features['wp_price_shock'],
                "b_wp_triple_interaction": b_wp_features['wp_triple_interaction'],
                # Target

                "target": b_t_plus_1,
            })

    return pd.DataFrame(rows)


# ============================================================================
# 예측 함수 (32개 피처 + ⭐ LGBM/XGB 앙상블)
# ============================================================================

def predict_optimized(pivot, pivot_weight, pairs, model_lgb, model_xgb, months_dt):
    """
    32개 피처 기반 예측 + LGBM 70% + XGB 30% 앙상블
    """
    months = months_dt
    n_months = len(months)
    t_last = n_months - 1
    t_prev = n_months - 2
    preds = []

    target_month = months[-1].month + 1 if months[-1].month < 12 else 1

    for row in tqdm(pairs.itertuples(index=False), desc="Predicting"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        a_weight_series = pivot_weight.loc[leader].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        if t_last - lag < 0 or t_last < 12:
            continue

        # Value 변수
        b_t = b_series[t_last]
        b_t_1 = b_series[t_prev]
        b_t_2 = b_series[t_prev - 1]
        b_t_12 = b_series[t_last - 12]
        a_t_lag = a_series[t_last - lag]

        # Weight 변수
        b_weight_t = b_weight_series[t_last]
        # 지난달 Weight (단가 변동률 계산용)
        b_weight_t_1 = b_weight_series[t_prev]
        b_weight_features = create_conditional_weight_features(
            value=b_t,
            weight=b_weight_t,
            value_series=b_series
        )

        b_price_features = create_enhanced_price_features_v2(
            value=b_t,
            weight=b_weight_t,
            value_series=b_series,
            weight_series=b_weight_series
        )
        b_wp_features = create_weight_price_interaction_features(
            value=b_t,
            weight=b_weight_t,
            value_series=b_series,
            weight_series=b_weight_series
        )

        # A 트렌드
        a_t_lag_diff = a_series[t_last - lag] - a_series[t_last - lag - 1]
        a_momentum = a_series[t_last - lag] / (a_series[t_last - lag - 3] + 1)
        a_roll_mean_3 = (a_series[t_last-lag] + a_series[t_last-lag-1] + a_series[t_last-lag-2]) / 3

        # B 트렌드
        b_diff = b_t - b_t_1
        b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
        b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
        b_roll_mean_3 = (b_series[t_last] + b_series[t_last-1] + b_series[t_last-2]) / 3
        b_roll_mean_6 = b_series[t_last-5:t_last+1].mean()
        b_roll_std_3 = np.std([b_series[t_last], b_series[t_last-1], b_series[t_last-2]])

        # 상호작용
        ab_ratio = a_t_lag / (b_t + 1)
        ab_diff = a_t_lag - b_t
        corr_weighted_a = a_t_lag * corr

        # 추가 고급 피처
        b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
        b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
        b_yoy_ratio = b_t / (b_t_12 + 1)
        b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
        recent_12 = b_series[max(0, t_last-11):t_last+1]
        b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

        # 계절성
        quarter = (target_month - 1) // 3 + 1
        is_year_end = 1 if target_month in [11, 12, 1] else 0

        # Lag
        lag_weight = 1 / (1 + lag)



        # 32개 피처 배열
        X_test = np.array([[
            b_t, b_t_1, b_t_12, a_t_lag, corr, float(lag), float(target_month),
            a_t_lag_diff, a_momentum, a_roll_mean_3,
            b_diff, b_pct_change, b_yoy_growth, b_roll_mean_3, b_roll_mean_6, b_roll_std_3,
            ab_ratio, ab_diff, corr_weighted_a,
            b_cv, b_acceleration, b_yoy_ratio, b_ma_ratio, b_percentile_rank,
            float(quarter), is_year_end,
            lag_weight,
            b_weight_features['weight_for_high_value'],
            b_weight_features['weight_value_ratio'],
            b_weight_features['exp_weighted'],
            b_weight_features['weight_importance'],
            # 🔥 Price 피처 5개 추가
            b_price_features['price_for_high_value'],
            b_price_features['price_value_interaction'],
            b_price_features['price_volatility_conditional'],
            b_price_features['price_trend_strength'],
            b_price_features['price_volume_pattern'],
            b_wp_features['wp_efficiency'],
            b_wp_features['wp_price_shock'],
            b_wp_features['wp_triple_interaction']
        ]])

        # ====================================================================
        # ⭐⭐ LGBM + XGB 앙상블 ⭐⭐
        # ====================================================================
        pred_lgb = model_lgb.predict(X_test)[0]
        pred_xgb = model_xgb.predict(X_test)[0]
        pred_lgb = np.maximum(0, pred_lgb)
        pred_xgb = np.maximum(0, pred_xgb)

        value_median = np.median(b_series[b_series > 0]) if (b_series > 0).any() else 0

        if b_t > value_median:
            # Value 클 때: LGBM 더 신뢰 (Weight 피처 강함)
            weight_lgb = 0.75
            weight_xgb = 0.25
        else:
            # Value 작을 때: 균형
            weight_lgb = 0.65
            weight_xgb = 0.35

        # 70:30 가중 평균
        # 로그 스케일에서 평균을 구하고 다시 exp (큰 값에 휘둘리지 않음)
        y_pred = np.expm1(
            weight_lgb * np.log1p(pred_lgb) +
            weight_xgb * np.log1p(pred_xgb)
        )
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    return pd.DataFrame(preds)


# ============================================================================
# 메인 실행 (pairs_for_model이 이미 정의되어 있다고 가정)
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*80)
    print("공행성 쌍 확인")
    print("="*80)

    print(f"사용할 공행성 쌍: {len(pairs_for_model)}개")

    print("\n" + "="*80)
    print("학습 데이터 생성")
    print("="*80)

    df_train_model = build_training_data_optimized(
        pivot,
        pivot_weight,
        pairs_for_model,
        months_dt
    )

    print(f"✓ 생성된 학습 데이터 shape: {df_train_model.shape}")

    feature_cols = [
        'b_t', 'b_t_1', 'b_t_12', 'a_t_lag', 'max_corr', 'best_lag', 'month',
        'a_t_lag_diff', 'a_momentum', 'a_roll_mean_3',
        'b_diff', 'b_pct_change', 'b_yoy_growth',
        'b_roll_mean_3', 'b_roll_mean_6', 'b_roll_std_3',
        'ab_ratio', 'ab_diff', 'corr_weighted_a',
        'b_cv', 'b_acceleration', 'b_yoy_ratio', 'b_ma_ratio', 'b_percentile_rank',
        'quarter', 'is_year_end',
        'lag_weight',
        'b_weight_for_high_value',
        'b_weight_value_ratio',
        'b_exp_weighted',
        'b_weight_importance',
        # 🔥 개선된 Price 5개
        'b_price_for_high_value',
        'b_price_value_interaction',
        'b_price_volatility_conditional',
        'b_price_trend_strength',
        'b_price_volume_pattern',
        # 🔥 Weight-Price 상호작용 3개
        'b_wp_efficiency',
        'b_wp_price_shock',
        'b_wp_triple_interaction'

    ]

    print(f"총 피처 개수: {len(feature_cols)}개 (기존 28 + Weight 4)")

    # ========================================================================
    # 모델 학습 (LGBM + XGB 두 개 학습)
    # ========================================================================

    if df_train_model.empty:
        print("\n❌ 오류: 학습 데이터가 없습니다.")
        submission = pd.DataFrame(columns=['leading_item_id', 'following_item_id', 'value'])
    else:
        print("\n" + "="*80)
        print("모델 학습")
        print("="*80)

        train_X = df_train_model[feature_cols].values
        train_y = df_train_model["target"].values

        print(f"학습 샘플 수: {len(train_X):,}")
        print(f"피처 수: {len(feature_cols)}")

        # ⭐ LGBM 모델 학습
        model_lgb = LGBMRegressor(
            random_state=42,
            n_estimators=1000,
            learning_rate=0.008,
            max_depth=10,
            num_leaves=70,
            min_child_samples=15,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=0.05,
            min_split_gain=0.005,
            feature_fraction=0.9,
            bagging_freq=5,
            verbose=-1
        )

        print("LGBM 학습 중...")
        model_lgb.fit(train_X, train_y)
        print("✓ LGBM 학습 완료")

        # ⭐ XGBoost 모델 학습
        model_xgb = XGBRegressor(
            random_state=42,
            n_estimators=1200,
            learning_rate=0.015,
            max_depth=10,
            min_child_weight=3,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=0.05,
            gamma=0.01,
            tree_method='hist',
            verbosity=0
        )

        print("XGBoost 학습 중...")
        model_xgb.fit(train_X, train_y)
        print("✓ XGBoost 학습 완료")

        # ====================================================================
        # 피처 중요도
        # ====================================================================

        print("\n" + "="*80)
        print("피처 중요도 Top 10 (LGBM 기준)")
        print("="*80)

        feature_importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': model_lgb.feature_importances_
        }).sort_values('importance', ascending=False)

        print(feature_importance.head(10).to_string(index=False))

        weight_features = feature_importance[
            feature_importance['feature'].str.contains('weight')
        ]
        print("\nWeight 관련 피처:")
        print(weight_features.to_string(index=False))

        # ====================================================================
        # 예측
        # ====================================================================

        print("\n" + "="*80)
        print("2025년 8월 예측 (LGBM 70% + XGB 30%)")
        print("="*80)

        submission = predict_optimized(
            pivot,
            pivot_weight,
            pairs_for_model,
            model_lgb,
            model_xgb,
            months_dt
        )

        print(f"\n✓ 예측 완료: {len(submission)}개 쌍")
        print("\n상위 10개:")
        print(submission.head(10).to_string(index=False))

    # ========================================================================
    # 결과 저장
    # ========================================================================

    print("\n" + "="*80)
    print("결과 저장")
    print("="*80)

    date_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    submission.to_csv('sy_weight_ensemble_{datestr}.csv', index=False)

    print(f"✓ 예측 완료: 32개 피처 기반 + LGBM 70% + XGB 30% 앙상블")
    print("="*80)

2. train.csv보면 value 분포가 평균 1.7M, 표준편차 5.4M
75% : 1.0M, 90% : 3.8M, 99%: 32M, max 111M
-> 극도로 오른쪽으로 치우친 분포
지금은: 모델은 원시 target(target=b_{t+1})으로 학습
예측 때만 log1p-expm1로 앙상블
이 구조보다 애초에 모델을 log1p(target)로 학습시키는 게 훨씬 안정적
#0.3736594216

In [ ]:
"""
무역 데이터 공행성 예측 모델 (Weight 피처 포함)
- 기존 28개 피처 + 조건부 Weight 4개 = 총 32개 피처
- 검증 결과: Value 클 때 Weight 유용 (상관 0.640)
- ⭐ 최종 예측: LGBM 70% + XGBoost 30% 앙상블 ⭐
"""

import datetime
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor       # ⭐ 추가됨
from tqdm import tqdm

# ============================================================================
# 데이터 로드 및 전처리
# ============================================================================

print("="*80)
print("데이터 로드 시작")
print("="*80)

# 월별 집계 (Value + Weight)
monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)
    .agg({
        'value': 'sum',
        'weight': 'sum'
    })
)

monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

# Value pivot
pivot = (
    monthly
    .pivot(index="item_id", columns="ym", values="value")
    .fillna(0.0)
)

# Weight pivot (조건부 피처용)
pivot_weight = (
    monthly
    .pivot(index="item_id", columns="ym", values="weight")
    .fillna(0.0)
)

months_dt = pivot.columns.to_list()

print(f"✓ 데이터 로드 완료")
print(f"  - 품목 수: {len(pivot)}")
print(f"  - 기간: {len(months_dt)}개월")
print(f"  - Value pivot: {pivot.shape}")
print(f"  - Weight pivot: {pivot_weight.shape}")

# ============================================================================
# 조건부 Weight 피처 함수
# ============================================================================

def create_conditional_weight_features(value, weight, value_series):
    """
    검증 결과 기반 조건부 Weight 피처 생성

    검증 결과:
    - Value 작을 때: Weight-Value 상관 0.254 (약함)
    - Value 클 때: Weight-Value 상관 0.640 (강함)
    → Value 클 때만 Weight 강조!
    """
    value_median = np.median(value_series[value_series > 0]) if (value_series > 0).any() else 0
    value_q75 = np.percentile(value_series[value_series > 0], 75) if (value_series > 0).any() else 0

    # 1. Value 클 때만 Weight 강조
    if value > value_median:
        weight_for_high_value = weight * 1.5
    else:
        weight_for_high_value = 0

    # 2. Value 비례 가중치
    weight_value_ratio = weight * (value / (value_median + 1))

    # 3. 지수 가중 (Value 클수록 증가)
    exp_weighted = weight * (1 - np.exp(-value / (value_median + 1)))

    # 4. 구간별 차등 가중치
    if value > value_q75:
        weight_importance = weight * 1.5  # 상위 25%
    elif value > value_median:
        weight_importance = weight * 1.0  # 중간
    else:
        weight_importance = weight * 0.1  # 하위

    return {
        'weight_for_high_value': weight_for_high_value,
        'weight_value_ratio': weight_value_ratio,
        'exp_weighted': exp_weighted,
        'weight_importance': weight_importance
    }
def create_enhanced_price_features_v2(value, weight, value_series, weight_series):
    """
    Weight 피처의 성공 패턴을 Price에도 적용
    """
    eps = 1.0
    price_t = value / (weight + eps)

    past_prices = []
    for i in range(len(value_series)):
        if weight_series[i] > 0:
            past_prices.append(value_series[i] / (weight_series[i] + eps))

    if len(past_prices) == 0:
        return {
            'price_for_high_value': 0,
            'price_value_interaction': 0,
            'price_volatility_conditional': 0,
            'price_trend_strength': 0,
            'price_volume_pattern': 0
        }

    past_prices = np.array(past_prices)

    # Value 기준 통계
    value_median = np.median(value_series[value_series > 0]) if (value_series > 0).any() else 0
    value_q75 = np.percentile(value_series[value_series > 0], 75) if (value_series > 0).any() else 0

    # Price 기준 통계
    price_mean = np.mean(past_prices)
    price_std = np.std(past_prices)
    price_median = np.median(past_prices)

    # 1. Value 클 때만 Price 강조
    if value > value_median:
        price_for_high_value = price_t * 2.0
    else:
        price_for_high_value = 0

    # 2. Price-Value 상호작용 (조건부)
    if value > value_q75:
        price_value_interaction = (price_t / (price_mean + 0.01)) * (value / (value_median + 1)) * 2.0
    elif value > value_median:
        price_value_interaction = (price_t / (price_mean + 0.01)) * (value / (value_median + 1))
    else:
        price_value_interaction = 0

    # 3. 조건부 가격 변동성
    price_cv = price_std / (price_mean + 0.01)
    if value > value_median:
        price_volatility_conditional = price_cv * (value / value_median)
    else:
        price_volatility_conditional = 0

    # 4. 가격 트렌드 강도
    if len(past_prices) >= 6:
        recent_6 = past_prices[-6:]
        x = np.arange(len(recent_6))
        slope = np.polyfit(x, recent_6, 1)[0]
        price_trend_strength = slope / (price_mean + 0.01)
    else:
        price_trend_strength = 0

    # 5. 가격-거래량 패턴
    weight_median = np.median(weight_series[weight_series > 0]) if (weight_series > 0).any() else 1

    if price_t > price_median and weight < weight_median and value > value_median:
        price_volume_pattern = (price_t / price_median) * (value / value_median) / (weight / weight_median + 0.1)
    elif price_t < price_median and weight > weight_median and value < value_median:
        price_volume_pattern = -0.5
    else:
        price_volume_pattern = 0

    return {
        'price_for_high_value': price_for_high_value,
        'price_value_interaction': price_value_interaction,
        'price_volatility_conditional': price_volatility_conditional,
        'price_trend_strength': price_trend_strength,
        'price_volume_pattern': price_volume_pattern
    }

def create_weight_price_interaction_features(value, weight, value_series, weight_series):
    """
    Weight와 Price의 결합 피처
    """
    eps = 1.0
    price_t = value / (weight + eps)

    value_median = np.median(value_series[value_series > 0]) if (value_series > 0).any() else 0
    weight_median = np.median(weight_series[weight_series > 0]) if (weight_series > 0).any() else 1

    past_prices = []
    for i in range(len(value_series)):
        if weight_series[i] > 0:
            past_prices.append(value_series[i] / (weight_series[i] + eps))
    price_mean = np.mean(past_prices) if len(past_prices) > 0 else 1

    # 1. Value 대비 Weight-Price 효율성
    if value > value_median:
        efficiency = (value / (value_median + 1)) * (weight / (weight_median + 1)) * (price_t / (price_mean + 0.01))
    else:
        efficiency = 0

    # 2. 단가 급변 신호
    if len(past_prices) >= 3:
        recent_3 = past_prices[-3:]

        price_acceleration = (recent_3[-1] - recent_3[-2]) - (recent_3[-2] - recent_3[-3])
        price_shock = abs(price_acceleration) / (price_mean + 0.01)

    else:
        price_shock = 0

    # 3. Value-Weight-Price 삼중 상호작용
    if value > value_median and weight > weight_median and price_t > price_mean:
        triple_interaction = (value / value_median) * (weight / weight_median) * (price_t / price_mean)
    else:
        triple_interaction = 0

    return {
        'wp_efficiency': efficiency,
        'wp_price_shock': price_shock,
        'wp_triple_interaction': triple_interaction
    }
# ============================================================================
# 학습 데이터 구축 (32개 피처)
# ============================================================================

def build_training_data_optimized(pivot, pivot_weight, pairs, months_dt):
    """
    기존 28개 피처 + 조건부 Weight 4개 = 총 32개 피처
    """
    months = months_dt
    n_months = len(months)
    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        # Pivot에 존재하는지 확인
        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        # Value 시계열
        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # Weight 시계열
        a_weight_series = pivot_weight.loc[leader].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        start_t = max(lag+3, 12)

        for t in range(start_t, n_months - 1):
            # Value 변수
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            b_t_2 = b_series[t - 2]
            b_t_12 = b_series[t - 12]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]

            # Weight 변수
            b_weight_t = b_weight_series[t]

            target_month = months[t + 1].month

            # ========== 기존 28개 피처 ==========

            # 1. 선행 품목(A) 트렌드 (3개)
            a_t_lag_diff = a_series[t - lag] - a_series[t - lag - 1]
            a_momentum = a_series[t - lag] / (a_series[t - lag - 3] + 1)
            a_roll_mean_3 = (a_series[t-lag] + a_series[t-lag-1] + a_series[t-lag-2]) / 3

            # 2. 후행 품목(B) 트렌드 (6개)
            b_diff = b_t - b_t_1
            b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
            b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
            b_roll_mean_3 = (b_series[t] + b_series[t-1] + b_series[t-2]) / 3
            b_roll_mean_6 = (b_series[t-5:t+1]).mean()
            b_roll_std_3 = np.std([b_series[t], b_series[t-1], b_series[t-2]])

            # 3. A-B 상호작용 (3개)
            ab_ratio = a_t_lag / (b_t + 1)
            ab_diff = a_t_lag - b_t
            corr_weighted_a = a_t_lag * corr

            # 4. 추가 고급 피처 (5개)
            b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
            b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
            b_yoy_ratio = b_t / (b_t_12 + 1)
            b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
            recent_12 = b_series[max(0, t-11):t+1]
            b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

            # 5. 계절성 (2개)
            quarter = (target_month - 1) // 3 + 1
            is_year_end = 1 if target_month in [11, 12, 1] else 0

            # 6. Lag 가중치 (1개)
            lag_weight = 1 / (1 + lag)

            # ========== 조건부 Weight 피처 (4개) ==========
            b_weight_features = create_conditional_weight_features(
                value=b_t,
                weight=b_weight_t,
                value_series=b_series
            )
            #개선된 price 5개
            b_price_features = create_enhanced_price_features_v2(
                value=b_t,
                weight=b_weight_t,
                value_series=b_series[:t+1],  # 현재까지의 이력
                weight_series=b_weight_series[:t+1]
            )
            b_wp_features = create_weight_price_interaction_features(
                value=b_t,
                weight=b_weight_t,
                value_series=b_series[:t+1],
                weight_series=b_weight_series[:t+1]
            )


            # 데이터 행 생성
            rows.append({
                # 기본 (7개)
                "b_t": b_t,
                "b_t_1": b_t_1,
                "b_t_12": b_t_12,
                "a_t_lag": a_t_lag,
                "max_corr": corr,
                "best_lag": float(lag),
                "month": float(target_month),

                # A 트렌드 (3개)
                "a_t_lag_diff": a_t_lag_diff,
                "a_momentum": a_momentum,
                "a_roll_mean_3": a_roll_mean_3,

                # B 트렌드 (6개)
                "b_diff": b_diff,
                "b_pct_change": b_pct_change,
                "b_yoy_growth": b_yoy_growth,
                "b_roll_mean_3": b_roll_mean_3,
                "b_roll_mean_6": b_roll_mean_6,
                "b_roll_std_3": b_roll_std_3,

                # 상호작용 (3개)
                "ab_ratio": ab_ratio,
                "ab_diff": ab_diff,
                "corr_weighted_a": corr_weighted_a,

                # 추가 고급 피처 (5개)
                "b_cv": b_cv,
                "b_acceleration": b_acceleration,
                "b_yoy_ratio": b_yoy_ratio,
                "b_ma_ratio": b_ma_ratio,
                "b_percentile_rank": b_percentile_rank,

                # 계절성 (2개)
                "quarter": float(quarter),
                "is_year_end": is_year_end,

                # Lag (1개)
                "lag_weight": lag_weight,

                # 조건부 Weight 4개
                "b_weight_for_high_value": b_weight_features['weight_for_high_value'],
                "b_weight_value_ratio": b_weight_features['weight_value_ratio'],
                "b_exp_weighted": b_weight_features['exp_weighted'],
                "b_weight_importance": b_weight_features['weight_importance'],
                # 🔥 Price V2 5개
                "b_price_for_high_value": b_price_features['price_for_high_value'],
                "b_price_value_interaction": b_price_features['price_value_interaction'],
                "b_price_volatility_conditional": b_price_features['price_volatility_conditional'],
                "b_price_trend_strength": b_price_features['price_trend_strength'],
                "b_price_volume_pattern": b_price_features['price_volume_pattern'],
                # 🔥 WP 상호작용 3개
                "b_wp_efficiency": b_wp_features['wp_efficiency'],
                "b_wp_price_shock": b_wp_features['wp_price_shock'],
                "b_wp_triple_interaction": b_wp_features['wp_triple_interaction'],
                # Target

                "target": b_t_plus_1,
            })

    return pd.DataFrame(rows)


# ============================================================================
# 예측 함수 (32개 피처 + ⭐ LGBM/XGB 앙상블)
# ============================================================================

def predict_optimized(pivot, pivot_weight, pairs, model_lgb, model_xgb, months_dt):
    """
    32개 피처 기반 예측 + LGBM 70% + XGB 30% 앙상블
    """
    months = months_dt
    n_months = len(months)
    t_last = n_months - 1
    t_prev = n_months - 2
    preds = []

    target_month = months[-1].month + 1 if months[-1].month < 12 else 1

    for row in tqdm(pairs.itertuples(index=False), desc="Predicting"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        a_weight_series = pivot_weight.loc[leader].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        if t_last - lag - 3 < 0 or t_last < 12:
            continue

        # Value 변수
        b_t = b_series[t_last]
        b_t_1 = b_series[t_prev]
        b_t_2 = b_series[t_prev - 1]
        b_t_12 = b_series[t_last - 12]
        a_t_lag = a_series[t_last - lag]

        # Weight 변수
        b_weight_t = b_weight_series[t_last]
        # 지난달 Weight (단가 변동률 계산용)

        b_weight_features = create_conditional_weight_features(
            value=b_t,
            weight=b_weight_t,
            value_series=b_series
        )

        b_price_features = create_enhanced_price_features_v2(
            value=b_t,
            weight=b_weight_t,
            value_series=b_series,
            weight_series=b_weight_series
        )
        b_wp_features = create_weight_price_interaction_features(
            value=b_t,
            weight=b_weight_t,
            value_series=b_series,
            weight_series=b_weight_series
        )

        # A 트렌드
        a_t_lag_diff = a_series[t_last - lag] - a_series[t_last - lag - 1]
        a_momentum = a_series[t_last - lag] / (a_series[t_last - lag - 3] + 1)
        a_roll_mean_3 = (a_series[t_last-lag] + a_series[t_last-lag-1] + a_series[t_last-lag-2]) / 3

        # B 트렌드
        b_diff = b_t - b_t_1
        b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
        b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
        b_roll_mean_3 = (b_series[t_last] + b_series[t_last-1] + b_series[t_last-2]) / 3
        b_roll_mean_6 = b_series[t_last-5:t_last+1].mean()
        b_roll_std_3 = np.std([b_series[t_last], b_series[t_last-1], b_series[t_last-2]])

        # 상호작용
        ab_ratio = a_t_lag / (b_t + 1)
        ab_diff = a_t_lag - b_t
        corr_weighted_a = a_t_lag * corr

        # 추가 고급 피처
        b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
        b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
        b_yoy_ratio = b_t / (b_t_12 + 1)
        b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
        recent_12 = b_series[max(0, t_last-11):t_last+1]
        b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

        # 계절성
        quarter = (target_month - 1) // 3 + 1
        is_year_end = 1 if target_month in [11, 12, 1] else 0

        # Lag
        lag_weight = 1 / (1 + lag)



        # 32개 피처 배열
        X_test = np.array([[
            b_t, b_t_1, b_t_12, a_t_lag, corr, float(lag), float(target_month),
            a_t_lag_diff, a_momentum, a_roll_mean_3,
            b_diff, b_pct_change, b_yoy_growth, b_roll_mean_3, b_roll_mean_6, b_roll_std_3,
            ab_ratio, ab_diff, corr_weighted_a,
            b_cv, b_acceleration, b_yoy_ratio, b_ma_ratio, b_percentile_rank,
            float(quarter), is_year_end,
            lag_weight,
            b_weight_features['weight_for_high_value'],
            b_weight_features['weight_value_ratio'],
            b_weight_features['exp_weighted'],
            b_weight_features['weight_importance'],
            # 🔥 Price 피처 5개 추가
            b_price_features['price_for_high_value'],
            b_price_features['price_value_interaction'],
            b_price_features['price_volatility_conditional'],
            b_price_features['price_trend_strength'],
            b_price_features['price_volume_pattern'],
            b_wp_features['wp_efficiency'],
            b_wp_features['wp_price_shock'],
            b_wp_features['wp_triple_interaction']
        ]])

        # ====================================================================
        # ⭐⭐ LGBM + XGB 앙상블 ⭐⭐
        # ====================================================================
        pred_lgb_log = float(model_lgb.predict(X_test)[0])
        pred_xgb_log = float(model_xgb.predict(X_test)[0])

        # 필요하다면 극단값 클리핑도 가능 (옵션)
        # pred_lgb_log = np.clip(pred_lgb_log, -5, 30)
        # pred_xgb_log = np.clip(pred_xgb_log, -5, 30)

        value_median = np.median(b_series[b_series > 0]) if (b_series > 0).any() else 0

        if b_t > value_median:
            # Value 클 때: LGBM 더 신뢰 (Weight 피처 강함)
            weight_lgb = 0.75
            weight_xgb = 0.25
        else:
            # Value 작을 때: 균형
            weight_lgb = 0.65
            weight_xgb = 0.35

        # 70:30 가중 평균
        # 로그 스케일에서 평균을 구하고 다시 exp (큰 값에 휘둘리지 않음)
        log_pred = weight_lgb * pred_lgb_log + weight_xgb * pred_xgb_log
        y_pred = np.expm1(log_pred)
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    return pd.DataFrame(preds)


# ============================================================================
# 메인 실행 (pairs_for_model이 이미 정의되어 있다고 가정)
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*80)
    print("공행성 쌍 확인")
    print("="*80)

    print(f"사용할 공행성 쌍: {len(pairs_for_model)}개")

    print("\n" + "="*80)
    print("학습 데이터 생성")
    print("="*80)

    df_train_model = build_training_data_optimized(
        pivot,
        pivot_weight,
        pairs_for_model,
        months_dt
    )

    print(f"✓ 생성된 학습 데이터 shape: {df_train_model.shape}")

    feature_cols = [
        'b_t', 'b_t_1', 'b_t_12', 'a_t_lag', 'max_corr', 'best_lag', 'month',
        'a_t_lag_diff', 'a_momentum', 'a_roll_mean_3',
        'b_diff', 'b_pct_change', 'b_yoy_growth',
        'b_roll_mean_3', 'b_roll_mean_6', 'b_roll_std_3',
        'ab_ratio', 'ab_diff', 'corr_weighted_a',
        'b_cv', 'b_acceleration', 'b_yoy_ratio', 'b_ma_ratio', 'b_percentile_rank',
        'quarter', 'is_year_end',
        'lag_weight',
        'b_weight_for_high_value',
        'b_weight_value_ratio',
        'b_exp_weighted',
        'b_weight_importance',
        # 🔥 개선된 Price 5개
        'b_price_for_high_value',
        'b_price_value_interaction',
        'b_price_volatility_conditional',
        'b_price_trend_strength',
        'b_price_volume_pattern',
        # 🔥 Weight-Price 상호작용 3개
        'b_wp_efficiency',
        'b_wp_price_shock',
        'b_wp_triple_interaction'

    ]

    print(f"총 피처 개수: {len(feature_cols)}개 (기존 28 + Weight 4)")

    # ========================================================================
    # 모델 학습 (LGBM + XGB 두 개 학습)
    # ========================================================================

    if df_train_model.empty:
        print("\n❌ 오류: 학습 데이터가 없습니다.")
        submission = pd.DataFrame(columns=['leading_item_id', 'following_item_id', 'value'])
    else:
        print("\n" + "="*80)
        print("모델 학습")
        print("="*80)

        train_X = df_train_model[feature_cols].values

        train_y_raw = df_train_model["target"].values
        train_y = np.log1p(train_y_raw)

        print(f"학습 샘플 수: {len(train_X):,}")
        print(f"피처 수: {len(feature_cols)}")

        # ⭐ LGBM 모델 학습
        model_lgb = LGBMRegressor(
            random_state=42,
            n_estimators=1000,
            learning_rate=0.008,
            max_depth=10,
            num_leaves=70,
            min_child_samples=15,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=0.05,
            min_split_gain=0.005,
            feature_fraction=0.9,
            bagging_freq=5,
            verbose=-1
        )

        print("LGBM 학습 중...")
        model_lgb.fit(train_X, train_y)
        print("✓ LGBM 학습 완료")

        # ⭐ XGBoost 모델 학습
        model_xgb = XGBRegressor(
            random_state=42,
            n_estimators=1200,
            learning_rate=0.015,
            max_depth=10,
            min_child_weight=3,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=0.05,
            gamma=0.01,
            tree_method='hist',
            verbosity=0
        )

        print("XGBoost 학습 중...")
        model_xgb.fit(train_X, train_y)
        print("✓ XGBoost 학습 완료")

        # ====================================================================
        # 피처 중요도
        # ====================================================================

        print("\n" + "="*80)
        print("피처 중요도 Top 10 (LGBM 기준)")
        print("="*80)

        feature_importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': model_lgb.feature_importances_
        }).sort_values('importance', ascending=False)

        print(feature_importance.head(10).to_string(index=False))

        weight_features = feature_importance[
            feature_importance['feature'].str.contains('weight')
        ]
        print("\nWeight 관련 피처:")
        print(weight_features.to_string(index=False))

        # ====================================================================
        # 예측
        # ====================================================================

        print("\n" + "="*80)
        print("2025년 8월 예측 (LGBM 70% + XGB 30%)")
        print("="*80)

        submission = predict_optimized(
            pivot,
            pivot_weight,
            pairs_for_model,
            model_lgb,
            model_xgb,
            months_dt
        )

        print(f"\n✓ 예측 완료: {len(submission)}개 쌍")
        print("\n상위 10개:")
        print(submission.head(10).to_string(index=False))

    # ========================================================================
    # 결과 저장
    # ========================================================================

    print("\n" + "="*80)
    print("결과 저장")
    print("="*80)

    date_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    submission.to_csv(f'sy_weight_ensemble_{date_str}.csv', index=False)

    print(f"✓ 예측 완료: 32개 피처 기반 + LGBM 70% + XGB 30% 앙상블")
    print("="*80)